In [3]:
import pandas as pd
import numpy as np

In [4]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Narela_Delhi_DPCC_2024.xlsx")

In [5]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,338.0,196.0,235.0,123.0,234.0,296.0,98.0,100.0,69.0,174.0,284.0,281.0
1,2,341.0,226.0,151.0,153.0,219.0,169.0,164.0,150.0,94.0,174.0,293.0,274.0
2,3,339.0,NaN,161.0,173.0,306.0,153.0,115.0,65.0,100.0,160.0,370.0,256.0
3,4,375.0,NaN,159.0,150.0,302.0,212.0,94.0,62.0,72.0,205.0,388.0,130.0
4,5,296.0,211.0,94.0,160.0,314.0,304.0,96.0,54.0,54.0,141.0,385.0,149.0
5,6,325.0,121.0,122.0,175.0,290.0,205.0,98.0,60.0,112.0,75.0,374.0,196.0
6,7,369.0,151.0,187.0,196.0,363.0,241.0,50.0,40.0,63.0,92.0,394.0,247.0
7,8,361.0,162.0,195.0,207.0,311.0,238.0,53.0,53.0,77.0,162.0,396.0,310.0
8,9,281.0,152.0,143.0,218.0,215.0,178.0,85.0,49.0,87.0,161.0,385.0,177.0
9,10,261.0,306.0,176.0,253.0,251.0,163.0,156.0,56.0,97.0,126.0,345.0,214.0


In [6]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [7]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [8]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [9]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,338.0,196.00,235.0,123.000000,234.0,296.000000,98.0,60.085714,69.000000,174.0,284.000000,281.0
1,2,341.0,226.00,151.0,153.000000,219.0,169.000000,164.0,60.085714,94.000000,174.0,293.000000,274.0
2,3,339.0,196.75,161.0,173.000000,306.0,153.000000,115.0,65.000000,100.000000,160.0,370.000000,256.0
3,4,375.0,196.75,159.0,150.000000,302.0,212.000000,94.0,62.000000,72.000000,205.0,388.000000,130.0
4,5,296.0,211.00,94.0,160.000000,314.0,304.000000,96.0,54.000000,54.000000,141.0,385.000000,149.0
5,6,325.0,121.00,122.0,175.000000,290.0,205.000000,98.0,60.000000,112.000000,75.0,374.000000,196.0
6,7,369.0,151.00,187.0,196.000000,363.0,241.000000,50.0,40.000000,63.000000,92.0,394.000000,247.0
7,8,361.0,162.00,195.0,207.000000,311.0,238.000000,53.0,53.000000,77.000000,162.0,396.000000,310.0
8,9,281.0,152.00,143.0,218.000000,215.0,178.000000,85.0,49.000000,87.000000,161.0,385.000000,177.0
9,10,261.0,306.00,176.0,253.000000,251.0,163.000000,156.0,56.000000,97.000000,126.0,345.000000,214.0
